<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/03_deep_learning/neural_network_basics/experiment_weight_initialization_comparison_xavier_he_zero_random.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Zero Initialization

In [1]:
import numpy as np

# Inputs
X = np.array([
    [1, 2],
    [3, 4]
])

# Zero weights
W = np.zeros((2, 3))

# Forward pass
Z = np.dot(X, W)

print(Z)

[[0. 0. 0.]
 [0. 0. 0.]]


Random Initialization

In [2]:
W = np.random.randn(2, 3)

print(W)

[[-0.50500732 -0.39854509 -1.42867618]
 [-0.12486245 -2.06281006  0.95758325]]


Problem with Large Weights

In [3]:
W = np.random.randn(2, 3) * 100

Demonstration

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.array([1000, -1000])

print(sigmoid(x))

[1. 0.]


/tmp/ipykernel_30034/717217357.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


# Exploding Gradient Problem

If weights keep increasing:

W
[1]
W
[2]
W
[3]
...W
[n]

Values explode exponentially.

Training becomes unstable.

# Xavier Initialization (Glorot)

Best for:

Sigmoid,
Tanh

Goal:
Maintain activation variance across layers.

Xavier Formula
$$W \sim N\left(0, \frac{1}{n_{\text{in}}}\right)$$

Xavier Initialization Code

In [5]:
n_in = 4
n_out = 3

W = np.random.randn(
    n_in,
    n_out
) * np.sqrt(1 / n_in)

print(W)

[[ 0.01627396  0.67721233 -0.03027069]
 [-0.07977677  0.35653471  0.64417109]
 [ 0.0672383  -0.1640611  -0.67300918]
 [ 0.13502777 -0.84950499  0.11361154]]


# He Initialization

Best for:

ReLU,
Deep Networks

Most commonly used in modern deep learning.

He Formula
$$W \sim N\left(0, \frac{2}{n_{\text{in}}}\right)$$

He Initialization Code

In [6]:
n_in = 4
n_out = 3

W = np.random.randn(
    n_in,
    n_out
) * np.sqrt(2 / n_in)

print(W)

[[ 0.55075479  0.49057215 -0.04454603]
 [ 0.93350258 -0.79987967 -0.01679821]
 [-0.09705725  0.52674382 -1.2989162 ]
 [ 0.60939659 -0.40815496  0.24280024]]


Why He Works Better for ReLU

ReLU removes negative activations:

f(x)=max(0,x)

Variance reduces by half.

He compensates for this.

Experimental Comparison

In [7]:
import numpy as np

from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.initializers import (
    RandomNormal,
    GlorotUniform,
    HeNormal,
    Zeros
)

In [8]:
iris = load_iris()

X = iris.data

y = to_categorical(iris.target)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

Function to Build Model

In [11]:
def build_model(initializer):

    model = Sequential()

    model.add(
        Dense(
            32,
            activation="relu",
            kernel_initializer=initializer,
            input_shape=(4,)
        )
    )

    model.add(
        Dense(
            16,
            activation="relu",
            kernel_initializer=initializer
        )
    )

    model.add(
        Dense(
            3,
            activation="softmax"
        )
    )

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

Compare Initializers

In [12]:
initializers = {

    "Zeros":
        Zeros(),

    "Random":
        RandomNormal(),

    "Xavier":
        GlorotUniform(),

    "He":
        HeNormal()
}

results = {}

Training Loop

In [13]:
for name, initializer in initializers.items():

    print(f"\nTraining with {name}")

    model = build_model(initializer)

    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=16,
        verbose=0
    )

    loss, accuracy = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    results[name] = accuracy

    print("Accuracy:", accuracy)


Training with Zeros


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy: 0.30000001192092896

Training with Random
Accuracy: 1.0

Training with Xavier
Accuracy: 1.0

Training with He
Accuracy: 0.9333333373069763


Print Final Results

In [14]:
print("\nFinal Comparison")

for name, accuracy in results.items():

    print(name, ":", accuracy)


Final Comparison
Zeros : 0.30000001192092896
Random : 1.0
Xavier : 1.0
He : 0.9333333373069763
